# 02 — Baseline & Komparasi Model
**Fase 3** · Menjawab Rumusan Masalah 1

Empat model dilatih dengan konfigurasi TF-IDF dan `random_state` identik,
sehingga perbedaan skor benar-benar berasal dari algoritmanya.

**Accuracy bukan metrik utama.** Dengan komposisi 72,3% positif, model yang
selalu menebak "positif" langsung memperoleh akurasi 72,3% tanpa mempelajari
apa pun. Prioritas: recall kelas negatif → macro-F1 → F1 negatif.

In [1]:
%load_ext autoreload
%autoreload 2
import sys, pathlib
ROOT = pathlib.Path.cwd().parents[1] if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd
from ml.src.config import load_config
cfg = load_config()
pd.set_option('display.width', 140)
print('seed:', cfg['seed'], '| baseline referensi:', cfg['evaluation']['majority_baseline'])

seed: 42 | baseline referensi: 0.723


## 1. Urutan yang tidak boleh dibalik

1. **split stratified** — sebelum apa pun yang melihat data
2. **deduplikasi** — HANYA training set
3. **fit vectorizer** — HANYA pada training set

Mem-fit TF-IDF pada data penuh, atau mendeduplikasi test set, adalah kebocoran
data yang membatalkan seluruh angka di fase ini.

In [2]:
from ml.src.train import siapkan_data
d = siapkan_data(dedup=True)

  data berlabel        : 96,466 baris (100.000 - 3.534 netral)
  train / test         : 77,172 / 19,294
  dedup training       : 77,172 -> 45,614 (31,558 dibuang)


  fitur TF-IDF         : 30,000


## 2. Tabel hasil lengkap

4 model × 2 set evaluasi + baseline.

In [3]:
m = pd.read_csv(ROOT / 'docs/tables/model_metrics.csv')
m = m[m.dedup_training]
m[['model_name','eval_set','n_eval','macro_f1','recall_neg','f1_neg','accuracy','train_seconds']].round(4)

,model_name,eval_set,n_eval,macro_f1,recall_neg,f1_neg,accuracy,train_seconds
0,majority_baseline,full,19294,0.4196,0.0000,0.0000,0.7229,0.0021
1,majority_baseline,informative_ge5w,8696,0.3173,0.0000,0.0000,0.4648,0.0021
2,complement_nb,full,19294,0.9154,0.9557,0.8816,0.9288,0.0130
3,complement_nb,informative_ge5w,8696,0.8986,0.9716,0.9127,0.9005,0.0130
4,multinomial_nb,full,19294,0.9279,0.9400,0.8977,0.9406,0.0119
5,multinomial_nb,informative_ge5w,8696,0.8991,0.9613,0.9120,0.9008,0.0119
6,linear_svc,full,19294,0.9301,0.9168,0.8996,0.9433,0.4271
7,linear_svc,informative_ge5w,8696,0.9000,0.9332,0.9098,0.9010,0.4271
8,logistic_regression,full,19294,0.9336,0.9316,0.9051,0.9459,0.3276
9,logistic_regression,informative_ge5w,8696,0.9078,0.9536,0.9181,0.9089,0.3276


## 3. Baseline yang harus dikalahkan

Baseline mayoritas dihitung **sebelum** model apa pun dilatih. Model yang tidak
mengungguli macro-F1 baseline tidak mempelajari apa pun.

In [4]:
base = m[m.model_name=='majority_baseline']
display(base[['eval_set','accuracy','macro_f1','recall_neg']].round(4))
mdl = m[m.model_name!='majority_baseline']
b = base[base.eval_set=='full'].macro_f1.iloc[0]
print(f"seluruh model mengungguli baseline macro_f1={b:.4f}:",
      bool((mdl[mdl.eval_set=='full'].macro_f1 > b).all()))

,eval_set,accuracy,macro_f1,recall_neg
0,full,0.7229,0.4196,0.0
1,informative_ge5w,0.4648,0.3173,0.0


seluruh model mengungguli baseline macro_f1=0.4196: True


## 4. Gambar 10 — perbandingan model

In [5]:
from ml.src import viz_models
viz_models.gambar_10_perbandingan().round(4)

eval_set,full,informative_ge5w
model_name,,
complement_nb,0.9154,0.8986
multinomial_nb,0.9279,0.8991
linear_svc,0.9301,0.9000
logistic_regression,0.9336,0.9078


## 5. Gambar 11 — temuan metodologis utama

Selisih macro-F1 antara set `full` dan `informative_ge5w` mengukur seberapa
besar kinerja model bertumpu pada kata pujian pendek yang berulang — 39,9%
ulasan hanya berisi ≤2 kata dan hampir seluruhnya positif (Temuan 2).

In [6]:
viz_models.gambar_11_selisih().round(4)

eval_set,full,informative_ge5w,selisih
model_name,,,
complement_nb,0.9154,0.8986,0.0168
logistic_regression,0.9336,0.9078,0.0259
multinomial_nb,0.9279,0.8991,0.0288
linear_svc,0.9301,0.9000,0.0300


## 6. Gambar 12 — waktu latih

Diukur `time.perf_counter()`, bukan dikira-kira.

In [7]:
viz_models.gambar_12_waktu().round(4)

,train_seconds,predict_seconds
model_name,,
complement_nb,0.0130,0.0021
multinomial_nb,0.0119,0.0019
linear_svc,0.4271,0.0012
logistic_regression,0.3276,0.0011


## 7. Dampak deduplikasi training

Varian dengan dan tanpa deduplikasi dijalankan dengan konfigurasi identik.
Selisihnya adalah bukti kuantitatif pengaruh 32,1% duplikat teks.

In [8]:
pd.read_csv(ROOT / 'docs/tables/dedup_impact.csv').round(4)

,model_name,tanpa_dedup,dengan_dedup,selisih
0,complement_nb,0.9060,0.9154,0.0094
1,linear_svc,0.9291,0.9301,0.0009
2,logistic_regression,0.9283,0.9336,0.0053
3,majority_baseline,0.4196,0.4196,0.0000
4,multinomial_nb,0.9281,0.9279,-0.0002
